In [0]:
from pyspark.sql import functions as F

campaigns_silver = spark.table(
    "workspace.fashion_silver.marketing_campaigns"
)

print("Campaign records:", campaigns_silver.count())
campaigns_silver.printSchema()

Campaign records: 200
root
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_type: string (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- campaign_objective: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- campaign_budget: double (nullable = true)
 |-- target_segment: string (nullable = true)
 |-- campaign_status: string (nullable = true)
 |-- targeted_customers: integer (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- customers_reached: integer (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- conversions: integer (nullable = true)
 |-- actual_spend: double (nullable = true)
 |-- attributed_revenue: double (nullable = true)
 |-- reach_rate: double (nullable = true)
 |-- click_through_rate: double (nullable = true)
 |-- conversion_rate: double (nullable = true)
 |-- return_on_ad_spend: double (nullable = true)
 |-- cost_pe

In [0]:
campaign_overview = campaigns_silver.agg(
    F.countDistinct("campaign_id").alias("total_campaigns"),
    F.round(F.sum("campaign_budget"), 2).alias("total_budget"),
    F.round(F.sum("actual_spend"), 2).alias("total_spend"),
    F.round(F.sum("attributed_revenue"), 2).alias("total_attributed_revenue"),
    F.sum("impressions").alias("total_impressions"),
    F.sum("customers_reached").alias("total_customers_reached"),
    F.sum("clicks").alias("total_clicks"),
    F.sum("conversions").alias("total_conversions"),
    F.round(F.avg("return_on_ad_spend"), 2).alias("avg_roas"),
    F.round(F.avg("conversion_rate"), 2).alias("avg_conversion_rate")
)

campaign_overview.show(truncate=False)

+---------------+-------------+-------------+------------------------+-----------------+-----------------------+------------+-----------------+--------+-------------------+
|total_campaigns|total_budget |total_spend  |total_attributed_revenue|total_impressions|total_customers_reached|total_clicks|total_conversions|avg_roas|avg_conversion_rate|
+---------------+-------------+-------------+------------------------+-----------------+-----------------------+------------+-----------------+--------+-------------------+
|200            |4.644090778E7|3.888342219E7|1.0855581268E8          |13963225         |2766050                |774886      |27180            |2.75    |3.47               |
+---------------+-------------+-------------+------------------------+-----------------+-----------------------+------------+-----------------+--------+-------------------+



In [0]:
channel_performance = (
    campaigns_silver
    .groupBy("marketing_channel")
    .agg(
        F.countDistinct("campaign_id").alias("total_campaigns"),
        F.round(F.sum("actual_spend"), 2).alias("total_spend"),
        F.round(F.sum("attributed_revenue"), 2).alias("attributed_revenue"),
        F.sum("impressions").alias("total_impressions"),
        F.sum("clicks").alias("total_clicks"),
        F.sum("conversions").alias("total_conversions"),
        F.round(F.avg("return_on_ad_spend"), 2).alias("avg_roas"),
        F.round(F.avg("conversion_rate"), 2).alias("avg_conversion_rate")
    )
    .orderBy(F.desc("attributed_revenue"))
)

channel_performance.show(truncate=False)

+-----------------+---------------+-----------+------------------+-----------------+------------+-----------------+--------+-------------------+
|marketing_channel|total_campaigns|total_spend|attributed_revenue|total_impressions|total_clicks|total_conversions|avg_roas|avg_conversion_rate|
+-----------------+---------------+-----------+------------------+-----------------+------------+-----------------+--------+-------------------+
|Email            |38             |8128474.14 |2.332607354E7     |2682485          |144335      |4817             |2.97    |3.5                |
|Instagram Ads    |40             |8782018.91 |1.96785077E7      |2583038          |126173      |5025             |2.34    |3.67               |
|Facebook Ads     |34             |5596383.27 |1.688267001E7     |2616676          |146505      |5038             |2.73    |3.49               |
|Google Ads       |30             |5743000.47 |1.634415349E7     |2715081          |163052      |5079             |2.82    |3.08  

In [0]:
campaign_type_performance = (
    campaigns_silver
    .groupBy("campaign_type")
    .agg(
        F.countDistinct("campaign_id").alias("total_campaigns"),
        F.round(F.sum("campaign_budget"), 2).alias("total_budget"),
        F.round(F.sum("actual_spend"), 2).alias("total_spend"),
        F.round(F.sum("attributed_revenue"), 2).alias("attributed_revenue"),
        F.sum("conversions").alias("total_conversions"),
        F.round(F.avg("return_on_ad_spend"), 2).alias("avg_roas"),
        F.round(F.avg("conversion_rate"), 2).alias("avg_conversion_rate")
    )
    .orderBy(F.desc("attributed_revenue"))
)

campaign_type_performance.show(truncate=False)

+------------------+---------------+-------------+-----------+------------------+-----------------+--------+-------------------+
|campaign_type     |total_campaigns|total_budget |total_spend|attributed_revenue|total_conversions|avg_roas|avg_conversion_rate|
+------------------+---------------+-------------+-----------+------------------+-----------------+--------+-------------------+
|Promotional       |45             |1.100103313E7|9011707.54 |2.83465098E7      |7065             |3.0     |3.75               |
|Seasonal          |49             |1.115021301E7|9556454.95 |2.765847748E7     |5869             |2.71    |3.09               |
|Festive           |45             |1.133595376E7|9377345.07 |2.403631356E7     |5567             |2.59    |3.45               |
|Product Launch    |35             |7432852.8    |6242821.48 |1.861784098E7     |4812             |2.87    |3.35               |
|Customer Retention|26             |5520855.08   |4695093.15 |9896670.86        |3867            

In [0]:
objective_performance = (
    campaigns_silver
    .groupBy("campaign_objective")
    .agg(
        F.countDistinct("campaign_id").alias("total_campaigns"),
        F.round(F.sum("actual_spend"), 2).alias("total_spend"),
        F.round(F.sum("attributed_revenue"), 2).alias("attributed_revenue"),
        F.sum("customers_reached").alias("total_customers_reached"),
        F.sum("conversions").alias("total_conversions"),
        F.round(F.avg("return_on_ad_spend"), 2).alias("avg_roas"),
        F.round(F.avg("conversion_rate"), 2).alias("avg_conversion_rate")
    )
    .orderBy(F.desc("attributed_revenue"))
)

objective_performance.show(truncate=False)

+--------------------+---------------+-------------+------------------+-----------------------+-----------------+--------+-------------------+
|campaign_objective  |total_campaigns|total_spend  |attributed_revenue|total_customers_reached|total_conversions|avg_roas|avg_conversion_rate|
+--------------------+---------------+-------------+------------------+-----------------------+-----------------+--------+-------------------+
|Sales Conversion    |72             |1.411467344E7|4.137875097E7     |1032286                |9460             |2.76    |3.55               |
|Customer Engagement |40             |7748226.32   |1.989154774E7     |534202                 |5374             |2.67    |3.41               |
|Brand Awareness     |37             |6264687.12   |1.717269924E7     |573475                 |5419             |2.72    |3.48               |
|Customer Acquisition|26             |5347075.58   |1.597120692E7     |290807                 |3595             |2.86    |3.46               |

In [0]:
target_segment_performance = (
    campaigns_silver
    .groupBy("target_segment")
    .agg(
        F.countDistinct("campaign_id").alias("total_campaigns"),
        F.round(F.sum("actual_spend"), 2).alias("total_spend"),
        F.round(F.sum("attributed_revenue"), 2).alias("attributed_revenue"),
        F.sum("customers_reached").alias("total_customers_reached"),
        F.sum("conversions").alias("total_conversions"),
        F.round(F.avg("return_on_ad_spend"), 2).alias("avg_roas"),
        F.round(F.avg("conversion_rate"), 2).alias("avg_conversion_rate")
    )
    .orderBy(F.desc("attributed_revenue"))
)

target_segment_performance.show(truncate=False)

+--------------------+---------------+------------+------------------+-----------------------+-----------------+--------+-------------------+
|target_segment      |total_campaigns|total_spend |attributed_revenue|total_customers_reached|total_conversions|avg_roas|avg_conversion_rate|
+--------------------+---------------+------------+------------------+-----------------------+-----------------+--------+-------------------+
|All Customers       |58             |1.09218801E7|2.698583108E7     |840906                 |7844             |2.5     |3.54               |
|New Customers       |42             |8604983.24  |2.463645207E7     |482069                 |4624             |2.8     |3.36               |
|Existing Customers  |44             |8290196.5   |2.407675811E7     |507445                 |4816             |2.86    |3.19               |
|Inactive Customers  |38             |7480628.34  |2.070398843E7     |566502                 |6834             |2.77    |3.71               |
|High 

In [0]:
campaign_performance = (
    campaigns_silver
    .select(
        "campaign_id",
        "campaign_name",
        "campaign_type",
        "marketing_channel",
        "campaign_objective",
        "target_segment",
        "campaign_status",
        "start_date",
        "end_date",
        "campaign_budget",
        "actual_spend",
        "impressions",
        "customers_reached",
        "clicks",
        "conversions",
        "attributed_revenue",
        "reach_rate",
        "click_through_rate",
        "conversion_rate",
        "return_on_ad_spend",
        "cost_per_conversion"
    )
    .orderBy(F.desc("return_on_ad_spend"))
)

campaign_performance.show(20, truncate=False)

+-----------+-----------------+------------------+-----------------+--------------------+--------------------+---------------+----------+----------+---------------+------------+-----------+-----------------+------+-----------+------------------+----------+------------------+---------------+------------------+-------------------+
|campaign_id|campaign_name    |campaign_type     |marketing_channel|campaign_objective  |target_segment      |campaign_status|start_date|end_date  |campaign_budget|actual_spend|impressions|customers_reached|clicks|conversions|attributed_revenue|reach_rate|click_through_rate|conversion_rate|return_on_ad_spend|cost_per_conversion|
+-----------+-----------------+------------------+-----------------+--------------------+--------------------+---------------+----------+----------+---------------+------------+-----------+-----------------+------+-----------+------------------+----------+------------------+---------------+------------------+-------------------+
|CMP000

In [0]:
campaign_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fashion_gold.campaign_performance")

In [0]:
channel_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fashion_gold.campaign_channel_performance")

In [0]:
campaign_type_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fashion_gold.campaign_type_performance")

In [0]:
objective_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fashion_gold.campaign_objective_performance")

In [0]:
target_segment_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fashion_gold.campaign_segment_performance")

In [0]:
campaign_overview.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fashion_gold.campaign_overview")

In [0]:
campaign_gold_tables = [
    "campaign_overview",
    "campaign_performance",
    "campaign_channel_performance",
    "campaign_type_performance",
    "campaign_objective_performance",
    "campaign_segment_performance"
]

for table_name in campaign_gold_tables:
    full_table_name = f"workspace.fashion_gold.{table_name}"
    print(f"{table_name}: {spark.table(full_table_name).count()} records")

campaign_overview: 1 records
campaign_performance: 200 records
campaign_channel_performance: 8 records
campaign_type_performance: 5 records
campaign_objective_performance: 5 records
campaign_segment_performance: 5 records
